# Check a YAML report by hand

Runs `powerpoint_input.yaml` through the loader and the **pptx** back end and
shows what comes out at every stage: the tasks, the finished deck, what it
*says* against the reference captured from the old `.rdf`, and what it
*shows* -- the pictures, the table and the videos a text comparison cannot
see. `test_pptx_generation.py` beside this notebook makes the same checks
unattended; this is the same thing spread out, so you can look at each step.

Pick the `.venv313` kernel of the `dev-yaml` worktree -- the first cell checks.

In [1]:
import subprocess
import sys
from pathlib import Path

import Scriptum
import yaml

# Derived from where this notebook sits, so moving the worktree does not
# turn the check below into a false alarm. Jupyter starts in the notebook's
# own directory, and this cell runs before anything changes it.
NOTEBOOK_DIR = Path.cwd()
WORKTREE = NOTEBOOK_DIR.parents[2]
here = Path(Scriptum.__file__).resolve().parent.parent

print('python    ', sys.executable)
print('Scriptum  ', Path(Scriptum.__file__).resolve())
print('PyYAML    ', yaml.__version__)
print('branch    ', subprocess.run(['git', 'branch', '--show-current'], cwd=here,
                                   capture_output=True, text=True).stdout.strip())

if here != WORKTREE:
    print()
    print('WARNING: Scriptum is NOT coming from the dev-yaml worktree.')
    print('         Expected', WORKTREE)
    print('         Pick the .venv313 kernel of that worktree - see the cell above.')
else:
    print()
    print('OK: importing from the dev-yaml worktree.')

python     E:\users\tel\Python\dev\Scriptum-Report-dev-yaml\.venv313\Scripts\python.exe
Scriptum   E:\users\tel\Python\dev\Scriptum-Report-dev-yaml\Scriptum\__init__.py
PyYAML     6.0.3
branch     dev-yaml

OK: importing from the dev-yaml worktree.


## A workspace

The run writes a deck and needs its data beside it, so everything is copied
to a temp directory. The repo stays clean, and you can throw the directory away.

In [2]:
import os
import shutil
import tempfile

REPORT_DIR = WORKTREE / 'tests' / '04_examples' / 'pptreport'
DATA_SOURCE = WORKTREE / 'tests' / 'data_source'

def workspace():
    """A fresh directory holding the fixtures, the template and the data."""
    work = Path(tempfile.mkdtemp(prefix='scriptum-'))
    for pattern in ('*.yaml', 'template.pptx'):
        for path in REPORT_DIR.glob(pattern):
            shutil.copy(path, work)
    shutil.copytree(DATA_SOURCE, work / 'data', dirs_exist_ok=True)
    os.chdir(work)
    return work

WORK = workspace()
print('working in', WORK)
print(sorted(p.name for p in WORK.iterdir()))

working in C:\Users\tel\AppData\Local\Temp\scriptum-ud80dzkq
['data', 'powerpoint_input.yaml', 'template.pptx']


## 1. Read the document

`ReportDataFile` reads a `.yaml` document through `Scriptum.rdf.loader`;
anything else is refused with a message. The `.rdf` text parser is gone.

A broken document raises `DocumentError` carrying **every** diagnostic, not
just the first.

In [3]:
rdf = Scriptum.ReportDataFile('powerpoint_input.yaml')

print('documenttype:', rdf.settings.documenttype)
print('datadir     :', rdf.settings.datadir)
print('tasks       :', len(rdf.tasks))
print('errors      :', rdf.errors or 'none')

documenttype: pptx
datadir     : data
tasks       : 69
errors      : none


## 2. What the tasks say

`what` is the operation -- for a deck every top-level entry is a `copy`: the
layout of that name is cloned into a new slide. `where` is the marker an *add*
lands at, `target` the placeholder or tag name in the layout, and the address
the **instance**: `:resultsgeneral::3` is the third slide made from the
`ResultsGeneral` layout.

Note the `_global_` tasks at the end: global fills are applied last, and the
task list carries that rule so no back end has to remember it.

In [4]:
def show_tasks(tasks, limit=None, only=None):
    rows = [t for t in tasks if only is None or only in '.'.join(t.myAddress)]
    print(f'{"#":>4}  {"what":6} {"where":18} {"target":22} address')
    print('-' * 110)
    for t in rows[:limit]:
        print(f'{t.serial:>4}  {t.what or "-":6} {t.where or "-":18} '
              f'{t.target or "-":22} {".".join(t.myAddress)}')
    if limit and len(rows) > limit:
        print(f'... {len(rows) - limit} more')

show_tasks(rdf.tasks, limit=40)

   #  what   where              target                 address
--------------------------------------------------------------------------------------------------------------
   1  copy   -                  -                      :titleslide::1
   2  -      -                  title                  :titleslide::1.:title::1
   3  -      -                  subtitle               :titleslide::1.:subtitle::1
   4  -      -                  image:main_model       :titleslide::1.image:main_model::1
   5  copy   -                  -                      :taskprojectdefinition::1
   6  -      -                  image:model_icon       :taskprojectdefinition::1.image:model_icon::1
   7  -      -                  manu                   :taskprojectdefinition::1.:manu::1
   8  -      -                  plat                   :taskprojectdefinition::1.:plat::1
   9  -      -                  hierarchy              :taskprojectdefinition::1.:hierarchy::1
  10  -      -                  proj_ver      

Try `show_tasks(rdf.tasks, only=':resultsgeneral')` to see the three slides
made from one layout -- the first is the empty entry (a bare slide), the
second carries the graph, the third the video.

In [5]:
show_tasks(rdf.tasks, only=':resultsgeneral')

   #  what   where              target                 address
--------------------------------------------------------------------------------------------------------------
  40  copy   -                  -                      :resultsgeneral::1
  41  copy   -                  -                      :resultsgeneral::2
  42  -      -                  title                  :resultsgeneral::2.:title::1
  43  -      -                  what                   :resultsgeneral::2.:what::1
  44  -      -                  plant                  :resultsgeneral::2.:plant::1
  45  add    marker:content::1  image:history          :resultsgeneral::2.image:history::1
  46  copy   -                  -                      :resultsgeneral::3
  47  -      -                  title                  :resultsgeneral::3.:title::1
  48  -      -                  what                   :resultsgeneral::3.:what::1
  49  add    marker:content::1  video:generic          :resultsgeneral::3.video:generic::1
  50

## 3. Build the deck

The steps mirror `common_case.run_pptx_case`, which every pptx case test
uses: `artist` fills the layouts, `remove_slide(0)` drops the template's own
first slide.

`finish=True` hands the saved deck to PowerPoint (Windows only) to re-save
it, and `createpdf=True` would export a PDF beside it; neither changes what
the deck says or shows, so both are off here as in the automated tests. Flip
`finish` on for a PowerPoint-saved copy -- it needs a PowerPoint that is not
already showing another `report.pptx`, and a deck PowerPoint does not open
in Protected View (it did for a deck in `%TEMP%`: "Automation rights are
not granted"); if it fails, the reason is printed below.

Anything the back end could not place prints a `WARNING`. A clean run prints
none.

In [6]:
import contextlib
import io as _io

with contextlib.redirect_stdout(_io.StringIO()) as printed:
    managed = Scriptum.ManagedPptx('template.pptx')
    managed.artist(rdf, directfill=True, globalfill=True,
                   cleardust=True, setproperties=True)
    managed.remove_slide(0)
    managed.save('report.pptx', finish=False, createpdf=False)

output = printed.getvalue().splitlines()
complaints = [line for line in output
              if 'WARNING' in line or 'ERROR' in line or 'failed' in line]
print('written:', WORK / 'report.pptx')
print('complaints:', len(complaints))
for line in complaints[:20]:
    print('  ', line)
if any('failed to update' in line for line in output):
    start = next(i for i, line in enumerate(output) if 'failed to update' in line)
    print('PowerPoint could not finish the deck:')
    for line in output[start:start + 6]:
        print('   ', line)

written: C:\Users\tel\AppData\Local\Temp\scriptum-ud80dzkq\report.pptx
complaints: 0


## 4. Read it back

Open `report.pptx` in PowerPoint if you want to look at it; this shows what it
*says* -- slide by slide, every text of every shape, then the cells of every
table -- which is what the automated comparison uses.

In [7]:
import pptx

def spoken(path):
    said = []
    for slide in pptx.Presentation(path).slides:
        for shape in slide.shapes:
            if shape.has_text_frame:
                for paragraph in shape.text_frame.paragraphs:
                    said.append(''.join(run.text for run in paragraph.runs).strip())
            if shape.has_table:
                for row in shape.table.rows:
                    said.extend(cell.text.strip() for cell in row.cells)
    return [line for line in said if line]

lines = spoken(WORK / 'report.pptx')
print(len(lines), 'non-empty lines')
for line in lines[:30]:
    print('  ', line[:100])

95 non-empty lines
   Test report
   Project: 4711
   23. Aug 2026 -- 18:24:58
   1234567
   who builds it
   where to built in
   don't know
   42
   238612
   Group X
   23. Aug 2026
   The manager
   0815
   4711
   ???
   -40
   many
   belongs to you
   where built
   N6676
   The initiator
   The designer
   The project manager
   Calc. done by
   The testplan manager
   his number
   her number
   its number
   wrong number
   who cares


## 5. Compare with the reference

`expected/powerpoint_input.json`, beside this notebook, is what this
fixture's **`.rdf`** produced before the back end changed (the `.rdf` and its
parser are gone; the reference is their record). Digits and weekday names
are collapsed on both sides, because the reference was captured on another
day and `date: now` is evaluated per run.

`IDENTICAL` below means the YAML document says exactly what the text one said.

In [8]:
import json
import re

DIGITS = re.compile(r'\d+')
WEEKDAY = re.compile(r'\b(?:Mon|Tue|Wed|Thu|Fri|Sat|Sun)\b')

def normalise(lines):
    return [WEEKDAY.sub('#', DIGITS.sub('#', line)) for line in lines]

REFERENCE = REPORT_DIR / 'expected' / 'powerpoint_input.json'

expected = normalise(json.loads(REFERENCE.read_text(encoding='utf-8')))
got = normalise(spoken(WORK / 'report.pptx'))

print(f'reference {len(expected)} lines, this run {len(got)} lines')
if expected == got:
    print('IDENTICAL')
else:
    for i, (a, b) in enumerate(zip(expected, got)):
        if a != b:
            print(f'first difference at line {i}')
            print('  reference:', a[:110])
            print('  this run :', b[:110])
            break
    else:
        print('one is a prefix of the other')

reference 95 lines, this run 95 lines
IDENTICAL


## 6. What it shows

A text comparison cannot see a picture, a table or a video -- the global
image fill of the docx back end placed nothing for a while and no comparison
noticed (`1f75367`). So: per slide, its layout and the shapes that are not
text. The sizes are what python-pptx computed from each file and its tag;
a picture placed at native size instead of the tag's shows up here.

In [9]:
from pptx.enum.shapes import MSO_SHAPE_TYPE
from pptx.util import Cm

def cm(length):
    return round(length / Cm(1), 2)

deck = pptx.Presentation(WORK / 'report.pptx')
for number, slide in enumerate(deck.slides, 1):
    print(f'slide {number}: {slide.slide_layout.name}')
    for shape in slide.shapes:
        kind = shape.shape_type
        if kind == MSO_SHAPE_TYPE.PICTURE:
            print(f'    picture  {cm(shape.width)} x {cm(shape.height)} cm  ({shape.image.content_type})')
        elif kind == MSO_SHAPE_TYPE.TABLE:
            print(f'    table    {len(shape.table.rows)} rows x {len(shape.table.columns)} columns')
        elif kind == MSO_SHAPE_TYPE.MEDIA:
            poster = shape.poster_frame.content_type if shape.poster_frame is not None else 'no poster'
            print(f'    video    {cm(shape.width)} x {cm(shape.height)} cm  ({shape.media_type}, poster {poster})')
        elif shape.has_text_frame and 'non existing image file' in shape.text_frame.text:
            print(f'    missing  {shape.text_frame.text.strip()[:80]}')

slide 1: TitleSlide
    picture  7.99 x 7.99 cm  (image/png)
slide 2: TaskProjectDefinition
    picture  3.4 x 3.4 cm  (image/png)
slide 3: Material
    table    4 rows x 5 columns
slide 4: ResultsGeneral
slide 5: ResultsGeneral
    picture  12.58 x 8.39 cm  (image/png)
slide 6: ResultsGeneral
    video    15.66 x 8.81 cm  (MOVIE (3), poster image/jpeg)
slide 7: ResultsAnimation
    video    11.5 x 6.47 cm  (MOVIE (3), poster image/jpeg)
slide 8: ResultsTwoBlocks
    missing  non existing image file 'data\\result1a.png'
    missing  non existing image file 'data\\result1b.png'
slide 9: BackCover


## 7. The verdict

The same expectations `test_pptx_generation.py` holds, so a manual run ends
with a yes or no: one picture each on the title slide, the definition slide
and the result slide, the 4 x 5 table on the material slide, a movie with a
poster frame on the two video slides, two pictures announced as missing on
the two-blocks slide, nothing on the back cover.

In [10]:
slides = list(deck.slides)
layouts = [slide.slide_layout.name for slide in slides]
assert layouts == ['TitleSlide', 'TaskProjectDefinition', 'Material',
                   'ResultsGeneral', 'ResultsGeneral', 'ResultsGeneral',
                   'ResultsAnimation', 'ResultsTwoBlocks', 'BackCover'], layouts
title, definition, material, empty, result, video, animation, two_blocks, back = slides

def of_kind(slide, kind):
    return [shape for shape in slide.shapes if shape.shape_type == kind]

def sizes(slide):
    return [(cm(p.width), cm(p.height)) for p in of_kind(slide, MSO_SHAPE_TYPE.PICTURE)]

assert sizes(title) == [(7.99, 7.99)], sizes(title)
assert sizes(definition) == [(3.4, 3.4)], sizes(definition)
assert sizes(result) == [(12.58, 8.39)], sizes(result)
assert sizes(empty) == [] and sizes(two_blocks) == []

(table,) = of_kind(material, MSO_SHAPE_TYPE.TABLE)
assert (len(table.table.rows), len(table.table.columns)) == (4, 5)

for slide in (video, animation):
    (movie,) = of_kind(slide, MSO_SHAPE_TYPE.MEDIA)
    assert movie.poster_frame is not None and movie.poster_frame.content_type == 'image/jpeg'

announced = [shape for shape in two_blocks.shapes
             if shape.has_text_frame and 'non existing image file' in shape.text_frame.text]
assert len(announced) == 2
assert len(back.shapes) == 0

print('all checks passed -- the deck shows what it should' if expected == got
      else 'shapes are right, but the text differs from the reference (see above)')

all checks passed -- the deck shows what it should
